# Trial Comparison Analysis: Mid-Air vs Button

This notebook analyzes and compares user performance between two conditions: **Mid-Air** and **Button**.

**Input Data Structure:**
- `midair/*.csv`: CSV files for mid-air interaction.
- `button/*.csv`: CSV files for button-based interaction.
- `TrailDefinitions.json`: Ground truth definitions.

**Analysis Metrics:**
1.  **MOD Error**: Mean variation from the ideal path.
2.  **Time Consumed**: Duration of each stroke.
3.  **Comparisons**: Overall and Per-Task Breakdown (by TrailTypeID).

In [ ]:
import pandas as pd
import numpy as np
import json
import glob
import os
import matplotlib.pyplot as plt
from scipy.spatial import KDTree
import seaborn as sns
from scipy import stats

# Set Style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Constants
CONDITIONS = ["midair", "button"]
DEFS_FILE = 'TrailDefinitions.json'

## 1. Load Definitions and Helpers

In [ ]:
# Load Definitions
if not os.path.exists(DEFS_FILE):
    print(f"Warning: {DEFS_FILE} not found. Please export it from Unity.")
    trace_defs = {}
else:
    with open(DEFS_FILE, 'r') as f:
        defs_json = json.load(f)
    trace_defs = {item['typeId']: item for item in defs_json['definitions']}
    print(f"Loaded {len(trace_defs)} definitions.")

def vector3_from_dict(d):
    return np.array([d['x'], d['y'], d['z']])

def generate_ideal_path(def_item, num_points=2000):
    start = vector3_from_dict(def_item['startPosition'])
    end = vector3_from_dict(def_item['endPosition'])
    
    shape = def_item['shape']
    amp_start = def_item['amplitudeStart']
    amp_end = def_item['amplitudeEnd']
    periods = def_item['periods']
    
    t_values = np.linspace(0, 1, num_points)
    points = []
    
    # Basis Calculation (Matches C#)
    baseline = end - start
    baseline_len = np.linalg.norm(baseline)
    baseline_dir = baseline / baseline_len if baseline_len > 0 else np.array([0,1,0])
    forward = np.array([0, 0, 1])
    wave_up = np.cross(baseline_dir, forward)
    norm_up = np.linalg.norm(wave_up)
    if norm_up < 0.001: wave_up = np.array([0, 1, 0])
    else: wave_up = wave_up / norm_up
        
    for t in t_values:
        straight_pos = start + (end - start) * t
        if shape == "Straight":
            points.append(straight_pos)
        else: 
            current_amp = amp_start + (amp_end - amp_start) * t
            sine_val = np.sin(t * periods * np.pi * 2.0)
            pos = straight_pos + wave_up * current_amp * sine_val
            points.append(pos)
    return np.array(points)

# Precompute Ideal Trees
ideal_trees = {}
ideal_points_cache = {}
for t_id, item in trace_defs.items():
    pts = generate_ideal_path(item)
    ideal_points_cache[t_id] = pts
    ideal_trees[t_id] = KDTree(pts)

## 2. Process Data

In [ ]:
all_stroke_data = []

for cond in CONDITIONS:
    search_path = os.path.join(cond, "*.csv")
    files = glob.glob(search_path)
    print(f"Found {len(files)} files in '{cond}'")
    
    for filepath in files:
        try:
            df = pd.read_csv(filepath)
            if 'StrokeID' not in df.columns: continue
            
            df = df.sort_values('Timestamp')
            grouped = df.groupby(['StrokeID', 'TargetID', 'TrailTypeID'])
            
            for (stroke_id, target_id, trail_type_id), group in grouped:
                if trail_type_id not in ideal_trees:
                    continue
                
                if len(group) < 5: continue
                
                # 1. Error Calc
                samples = group[['Position_X', 'Position_Y', 'Position_Z']].values
                tree = ideal_trees[trail_type_id]
                distances, _ = tree.query(samples)
                mean_error = np.mean(distances)
                
                # 2. Time Calc
                duration = group['Timestamp'].max() - group['Timestamp'].min()
                
                all_stroke_data.append({
                    'Condition': cond,
                    'StrokeID': stroke_id,
                    'TargetID': target_id,
                    'TrailTypeID': trail_type_id,
                    'Shape': trace_defs[trail_type_id]['shape'],
                    'Duration': duration,
                    'MOD_Error': mean_error
                })
        except Exception as e:
            print(f"Error processing {filepath}: {e}")

results_df = pd.DataFrame(all_stroke_data)
print(f"Processed {len(results_df)} total strokes.")

## 3. Visualization

Generating the three requested plots:
1.  Overall Error Comparison (Slim bars)
2.  Error Comparison by **Target Type ID**
3.  Overall Time Consumed (Slim bars)

In [ ]:
if not results_df.empty:
    fig, axes = plt.subplots(3, 1, figsize=(10, 18))

    # --- 1. Overall MOD Error (Slimmer) ---
    sns.barplot(data=results_df, x='Condition', y='MOD_Error', 
                errorbar='sd', capsize=.1, alpha=0.9, width=0.4, ax=axes[0], palette="viridis")
    axes[0].set_title('Overall MOD Error (Mean ± SD)', fontsize=16)
    axes[0].set_ylabel('Mean Euclidean Distance Error (m)', fontsize=12)
    axes[0].set_xlabel('')

    # --- 2. Per-Task Error (TrailTypeID) ---
    sns.barplot(data=results_df, x='TrailTypeID', y='MOD_Error', hue='Condition', 
                errorbar='sd', capsize=.1, alpha=0.9, ax=axes[1], palette="viridis")
    axes[1].set_title('MOD Error by Target Type (Mean ± SD)', fontsize=16)
    axes[1].set_ylabel('Mean Euclidean Distance Error (m)', fontsize=12)
    axes[1].set_xlabel('Target Type ID', fontsize=12)
    axes[1].legend(title='Condition', loc='upper right')

    # --- 3. Overall Time Consumed (Slimmer) ---
    sns.barplot(data=results_df, x='Condition', y='Duration', 
                errorbar='sd', capsize=.1, alpha=0.9, width=0.4, ax=axes[2], palette="viridis")
    axes[2].set_title('Overall Time Consumed (Mean ± SD)', fontsize=16)
    axes[2].set_ylabel('Duration (s)', fontsize=12)
    axes[2].set_xlabel('Condition', fontsize=12)

    plt.tight_layout()
    plt.show()
    
    # --- Summary Table ---
    print("--- Overall Stats ---")
    print(results_df.groupby(['Condition'])[['MOD_Error', 'Duration']].agg(['mean', 'std']).round(4))
else:
    print("No data loaded. Please check data folders.")

## 4. Statistical Analysis

We perform statistical tests to determine if the differences between 'midair' and 'button' are significant.

**Methodology:**
1.  **Normality Check**: Shapiro-Wilk test.
2.  **Significance Test**:
    -   If normal: **Independent t-test**.
    -   If non-normal: **Mann-Whitney U test**.

In [ ]:
def check_significance(df, metric, group_col='Condition'):
    print(f"\n=== Statistical Analysis for {metric} ===")
    
    groups = df[group_col].unique()
    if len(groups) != 2:
        print("Error: Need exactly 2 groups for comparison.")
        return
    
    g1 = df[df[group_col] == groups[0]][metric].dropna()
    g2 = df[df[group_col] == groups[1]][metric].dropna()
    
    print(f"Group 1 ({groups[0]}): n={len(g1)}, mean={g1.mean():.4f}, std={g1.std():.4f}")
    print(f"Group 2 ({groups[1]}): n={len(g2)}, mean={g2.mean():.4f}, std={g2.std():.4f}")
    
    # 1. Normality Check (Shapiro-Wilk)
    # For large samples (>5000), Shapiro might be too sensitive, but we'll use it as standard.
    _, p_norm1 = stats.shapiro(g1)
    _, p_norm2 = stats.shapiro(g2)
    
    is_normal = (p_norm1 > 0.05) and (p_norm2 > 0.05)
    print(f"Normality (p > 0.05): {groups[0]}={p_norm1:.4f}, {groups[1]}={p_norm2:.4f} -> {'Normal' if is_normal else 'Not Normal'}")
    
    # 2. Significance Test
    if is_normal:
        stat, p_val = stats.ttest_ind(g1, g2, equal_var=False) # Welch's t-test
        test_name = "Welch's t-test"
    else:
        stat, p_val = stats.mannwhitneyu(g1, g2)
        test_name = "Mann-Whitney U"
        
    print(f"Test: {test_name}")
    print(f"Statistic: {stat:.4f}, p-value: {p_val:.4e}")
    
    significance = "SIGNIFICANT" if p_val < 0.05 else "NOT SIGNIFICANT"
    print(f"Result: Difference is {significance} (alpha=0.05)")

if not results_df.empty:
    check_significance(results_df, 'MOD_Error')
    check_significance(results_df, 'Duration')
else:
    print("No data to analyze.")